In [1]:
import evidently

In [2]:
from evidently.ui.workspace import Workspace

# Create a workspace and assign it a name
ws = Workspace.create("mlops")

In [3]:
project = ws.create_project("Return Model Projekt")
project

Project ID: 019d85db-3cb2-78ca-ba63-4b180dd56ab0
Project Name: Return Model Projekt
Project Description: None
        

## Load Data

In [4]:
import pandas as pd

In [5]:
pd.read_csv("../data/train.csv")

,transactionId,basket,customerType,totalAmount,returnLabel
0,9534310106,"[4, 3, 4]",new,252.000000,1
1,7202594767,"[4, 2, 0, 2, 5]",existing,70.000000,0
2,2737331698,[5],existing,84.000000,0
3,4868011733,"[1, 4, 2, 4]",existing,116.000000,0
4,7622406570,"[2, 5, 3, 2, 3, 2, 0]",existing,378.000000,0
...,...,...,...,...,...
22395,5461363334,"[4, 5, 4]",new,147.000000,0
22396,9013779310,"[3, 4, 4, 4, 0]",existing,259.056014,1
22397,1590423615,"[0, 4, 3]",new,249.000000,1
22398,1800993941,"[0, 5, 1, 2, 5, 5, 4, 4]",existing,80.000000,0


In [6]:
train_df = pd.read_csv("../data/train.csv")
train_df["orderedBooks"] = train_df["basket"].apply(lambda x: sum(c.isdigit() for c in x))
train_df = pd.get_dummies(train_df, columns=["customerType"], dtype=int, drop_first=True)
train_df =  train_df[["totalAmount", "orderedBooks", "customerType_new"]]
train_df.head()

,totalAmount,orderedBooks,customerType_new
0,252.0,3,1
1,70.0,5,0
2,84.0,1,0
3,116.0,4,0
4,378.0,7,0


In [7]:
live_df = pd.read_csv("../data/live.csv")
print(live_df.shape)
live_df.dropna(inplace=True)
print(live_df.shape)
live_df =  live_df[["totalAmount", "orderedBooks", "customerType_new"]]
live_df

(9226, 6)
(9219, 6)


,totalAmount,orderedBooks,customerType_new
0,439.2,6.0,0.0
1,102.0,5.0,0.0
2,330.0,11.0,0.0
3,633.6,8.0,0.0
4,344.4,7.0,0.0
...,...,...,...
9221,79.2,2.0,0.0
9222,48.0,4.0,0.0
9223,151.2,1.0,0.0
9224,84.0,7.0,0.0


## Report for feature drift

In [8]:
from evidently import BinaryClassification, Dataset, DataDefinition

schema = DataDefinition(
    numerical_columns=["totalAmount", "orderedBooks"],
    categorical_columns=["customerType_new"]
)

train_data = Dataset.from_pandas(
    train_df,
    data_definition=schema
)

live_data = Dataset.from_pandas(
    train_df,
    data_definition=schema
)

In [9]:
from evidently import Report
from evidently.presets import DataDriftPreset

# Create a report with the DataDriftPreset
report = Report(metrics=[DataDriftPreset()])

# Run the report with your reference and current data
run = report.run(train_df, live_df)

In [10]:
ws.add_run(project.id, run)

Report ID: 019d85db-3fbd-7d5a-a959-0ab91f798903
Link: mlops/019d85db-3cb2-78ca-ba63-4b180dd56ab0/snapshots/019d85db-3fbd-7d5a-a959-0ab91f798903.json